# Lab 9: Object Detection 2, a tiny SSD

*Written by Yalda.*

In Lab 6 you built a tiny **YOLO**: one grid, one box per cell. Today we build a tiny **SSD** (Single Shot MultiBox Detector),
the detector of **Assignment 3**. SSD adds two ideas to YOLO:

1. **Default boxes** (also called *anchors*): every grid cell has a few boxes of different shapes. The network does not draw a box
   from nothing; it picks a default box and predicts a small **correction** to it.
2. **Several grids:** a fine grid for small objects, coarse grids for big objects.

We use a small version of the **Assignment 3 dataset** (cat, dog, person) and the same ideas, so every part of this lab is a
warm-up for one part of the assignment:

| Lab part | Assignment 3 section | Time |
|---|---|---|
| Setup + 1. A real SSD | 1. Introduction | 15 min |
| 2. The data | 4. (reading the dataset) | 15 min |
| 3. Default boxes | 3. Generating default bounding boxes | 20 min |
| 4. Matching and offsets | 4. Assigning ground truth objects to default boxes | 30 min |
| 5. The network | 5. The network | 20 min |
| 6. Loss and training | 6. The loss function | 20 min |
| 7. From outputs to boxes | 7. Non maximum suppression | 15 min |
| 8. Try it yourself | 10. Demo (run on test images) | 5 min |

**Same ideas, a smaller setup.** To train in a few minutes, the lab uses smaller images, different grid sizes and a
pretrained ResNet18, so the numbers here are different from the assignment.

For each **TODO**, write your code, then run the check cell below it. If you are stuck, open the hidden **Solution** cell.

## Environment (skip this on Colab)

On Colab everything is already installed. To run this notebook on your own machine or a lab machine, create a conda
environment **once** from a terminal (if you made the **vc** environment in Lab 7 or 8, you already have it):

```bash
conda create -n vc python=3.11 -y && conda activate vc && pip install torch torchvision numpy matplotlib pillow jupyter ipykernel && python -m ipykernel install --user --name vc
```

Then pick the **vc** kernel in Jupyter / VS Code (top right). On Windows with an NVIDIA GPU, get the install command
from [pytorch.org](https://pytorch.org/get-started/locally/) instead.

If you already have an environment, the next cell installs anything missing into the kernel you are using.
If it installs something, **restart the kernel** (*Kernel → Restart*), then continue.

In [ ]:
%pip install -q torch torchvision numpy matplotlib pillow

In [ ]:
# check your environment
import sys
print("python:", sys.version.split()[0], " from", sys.executable)
try:
    import torch
    import torchvision
    print("torch:", torch.__version__, " torchvision:", torchvision.__version__)
    print("NVIDIA GPU:", torch.cuda.is_available(), " Apple GPU:", torch.backends.mps.is_available())
except ImportError as e:
    print("ERROR:", e)
    print("Restart the kernel and run this cell again, or pick the vc kernel (see above).")

## Setup

In [ ]:
import os
import math
import random
import zipfile
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():   # Apple GPU
    device = "mps"
else:
    device = "cpu"
print("device:", device)

In [ ]:
# download the data (a small version of the A3 dataset, 18 MB) and a test photo
RAW = "https://raw.githubusercontent.com/arash-mham/visual-computing-I/main/lab-material/"
if not os.path.exists("coco_small"):
    if not os.path.exists("coco_small.zip"):
        urllib.request.urlretrieve(RAW + "lab9/coco_small.zip", "coco_small.zip")
    zipfile.ZipFile("coco_small.zip").extractall(".")
if not os.path.exists("sopranos_dinner.jpg"):
    urllib.request.urlretrieve(RAW + "lab9/sopranos_dinner.jpg", "sopranos_dinner.jpg")
print("train images:", len(os.listdir("coco_small/train/images")), " test images:", len(os.listdir("coco_small/test/images")))

## 1. A real SSD

torchvision comes with a real SSD (**SSDlite**, trained on COCO with 80 classes). Let's run it on a photo.

*(Still from The Sopranos, HBO, [CC BY 3.0](https://creativecommons.org/licenses/by/3.0/), via Wikimedia Commons.)*

In [ ]:
weights = torchvision.models.detection.SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
real_ssd = torchvision.models.detection.ssdlite320_mobilenet_v3_large(weights=weights).eval()   # downloads 13 MB
COCO_NAMES = weights.meta["categories"]

photo = Image.open("sopranos_dinner.jpg").convert("RGB")
photo_tensor = torchvision.transforms.functional.to_tensor(photo)     # (3, H, W), values 0..1
with torch.no_grad():
    result = real_ssd([photo_tensor])[0]

fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(photo)
for box, score, label in zip(result["boxes"], result["scores"], result["labels"]):
    if score > 0.3:
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="yellow", linewidth=2))
        ax.text(x1, y1 - 4, f"{COCO_NAMES[label]} {score:.2f}", color="yellow", fontsize=9, fontweight="bold")
ax.axis("off")
plt.show()

Where are the "several grids"? Let's look inside: the image is resized to 320×320, and the network makes feature maps
of **6 sizes**, so it has **6 grids**. Every cell of every grid has **6 default boxes** (6 shapes).

In the pictures: **red** = one cell, **yellow** = its 6 default boxes. Fine grids have small boxes (small objects),
coarse grids have big boxes (big objects).

In [ ]:
images, _ = real_ssd.transform([photo_tensor])                 # resize + normalize
feature_maps = list(real_ssd.backbone(images.tensors).values())
boxes_per_cell = real_ssd.anchor_generator.num_anchors_per_location()
total = 0
for f, k in zip(feature_maps, boxes_per_cell):
    g = f.shape[2]
    print(f"grid {g:2d} x {g:2d}: {g * g:3d} cells x {k} default boxes = {g * g * k:4d}")
    total += g * g * k
print("total default boxes:", total)

# draw every grid on the (resized) photo, with the default boxes of one cell near the middle
default_boxes = real_ssd.anchor_generator(images, feature_maps)[0]   # (3234, 4) corners, in pixels of the 320 x 320 image
small = photo.resize((320, 320))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
start = 0
for ax, f, k in zip(axes.flat, feature_maps, boxes_per_cell):
    g = f.shape[2]
    ax.imshow(small)
    for i in range(1, g):                                       # grid lines
        ax.axhline(i * 320 / g, color="white", linewidth=0.5)
        ax.axvline(i * 320 / g, color="white", linewidth=0.5)
    row, col = g // 2, g // 2
    first = start + (row * g + col) * k                         # the first default box of this cell
    for x1, y1, x2, y2 in default_boxes[first : first + k].tolist():
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="yellow", linewidth=1.5))
    ax.add_patch(patches.Rectangle((col * 320 / g, row * 320 / g), 320 / g, 320 / g, fill=False, edgecolor="red", linewidth=3))
    ax.set_xlim(0, 320)
    ax.set_ylim(320, 0)
    ax.set_title(f"{g} x {g} grid")
    ax.axis("off")
    start += g * g * k
plt.show()

So the real SSD checks **3,234** default boxes in one pass. Today we use 4 grids (8, 4, 2, 1)
with 4 boxes each: $(64 + 16 + 4 + 1) \times 4 = 340$.

| | YOLO (Lab 6) | SSD (today) |
|---|---|---|
| grids | one ($7 \times 7$) | several (8, 4, 2, 1) |
| boxes per cell | 1 | 4 default boxes of different shapes |
| what is predicted for a box | $x, y, w, h$ directly | a small **correction** (offset) to a default box |
| "is there an object?" | a separate `conf` number | a 4th class: **background** |
| one object is assigned to | one cell | **every** default box that overlaps it enough |

**Andrew Ng: Anchor Boxes (10 min)** (click to watch). He adds anchor boxes to YOLO. SSD's default boxes are the same idea.

<a href="https://www.youtube.com/watch?v=RTlwl2bv0Tg" target="_blank"><img src="https://img.youtube.com/vi/RTlwl2bv0Tg/hqdefault.jpg" width="480"></a>

## 2. The data

This is a small version of the assignment dataset: 2,740 training images of **cats, dogs and people**, made smaller
(longest side 160 pixels) so it downloads fast. The folders look like this:

```
coco_small/train/images/00000.jpg        coco_small/train/annotations/00000.txt
coco_small/test/images/00000.jpg         (the test set has no annotations)
```

Every line of an annotation file is one object: `class x_min y_min width height`, **in pixels**.
Classes: 0 = cat, 1 = dog, 2 = person.

In [ ]:
print(open("coco_small/train/annotations/00000.txt").read())
print("image size (width, height):", Image.open("coco_small/train/images/00000.jpg").size)

Images have different sizes, but the network wants 128×128. The trick: **divide** the box by the image size.
Then every number is between 0 and 1, and the box is still right after we resize the image to anything.
Example: point $(x = 100, y = 200)$ in an image of width 1000 and height 500 becomes $(0.1, 0.4)$.

We store boxes as **corners** `[x1, y1, x2, y2]` (normalized).

**TODO 1:** fill in `read_annotation`.

In [ ]:
def read_annotation(path, W, H):
    # returns boxes: (n, 4) array of [x1, y1, x2, y2], normalized to 0..1, and labels: list of n class numbers
    boxes = []
    labels = []
    for line in open(path):
        c, x, y, w, h = line.split()
        x, y, w, h = float(x), float(y), float(w), float(h)
        # TODO: the corners, divided by the image width W (x values) or the height H (y values)
        x1 = ...
        y1 = ...
        x2 = ...
        y2 = ...
        boxes.append([x1, y1, x2, y2])
        labels.append(int(c))
    return np.array(boxes, dtype=np.float32), labels

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def read_annotation(path, W, H):
    # returns boxes: (n, 4) array of [x1, y1, x2, y2], normalized to 0..1, and labels: list of n class numbers
    boxes = []
    labels = []
    for line in open(path):
        c, x, y, w, h = line.split()
        x, y, w, h = float(x), float(y), float(w), float(h)
        x1 = x / W
        y1 = y / H
        x2 = (x + w) / W
        y2 = (y + h) / H
        boxes.append([x1, y1, x2, y2])
        labels.append(int(c))
    return np.array(boxes, dtype=np.float32), labels

In [ ]:
# run this cell to check your code
boxes, labels = read_annotation("coco_small/train/annotations/00000.txt", 160, 120)
print(boxes, labels)
print("correct:", np.allclose(boxes, [[0.45675, 0.3123333, 0.686, 0.618]]) and labels == [0])

In [ ]:
DATA = "coco_small"
CLASSES = ["cat", "dog", "person"]
COLORS = ["tab:red", "tab:blue", "tab:green"]
IMG = 128                                     # the network input is IMG x IMG


def load(name):
    image = Image.open(f"{DATA}/train/images/{name}.jpg").convert("RGB")
    W, H = image.size
    boxes, labels = read_annotation(f"{DATA}/train/annotations/{name}.txt", W, H)
    image = np.array(image.resize((IMG, IMG)))   # (128, 128, 3); the boxes are normalized, so they still fit
    return image, boxes, labels


def draw_boxes(ax, boxes, labels, scores=None, linestyle="-", size=IMG):
    # boxes: corner format, normalized
    for i in range(len(boxes)):
        x1, y1, x2, y2 = [float(v) * size for v in boxes[i]]
        color = COLORS[int(labels[i])]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=3, linestyle=linestyle))
        text = CLASSES[int(labels[i])]
        if scores is not None:
            text += f" {float(scores[i]):.2f}"
        ax.text(x1, y1 - 2, text, color=color, fontsize=15, fontweight="bold")


# split 90% train / 10% validation
names = sorted(f[:-4] for f in os.listdir(f"{DATA}/train/images"))
n_val = len(names) // 10
val_names = names[:n_val]
train_names = names[n_val:]
print("train:", len(train_names), " validation:", len(val_names))

fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, name in zip(axes.flat, train_names):
    image, boxes, labels = load(name)
    ax.imshow(image)
    draw_boxes(ax, boxes, labels)
    ax.axis("off")
plt.tight_layout()
plt.show()

The images are squashed to a square, and the boxes still fit because they are normalized.

**Look closely:** some images have a second cat, dog or person **without a box**. This dataset labels mostly one object
per image (only about 8% of images have more than one box). Remember this later, when your model "finds" something that is not in the labels.

## 3. Default boxes (A3 section 3)

Every cell of every grid gets **4 default boxes**, all centered at the center of the cell. For grid number $i$ we use a small
size $s$ = `SMALL[i]` and a large size $l$ = `LARGE[i]`:

| box | width | height |
|---|---|---|
| small square | $s$ | $s$ |
| large square | $l$ | $l$ |
| wide | $l\sqrt{2}$ | $l / \sqrt{2}$ |
| tall | $l / \sqrt{2}$ | $l\sqrt{2}$ |

(The wide and tall boxes have the same area as the large square.) The center of cell (row, col) in a $g \times g$ grid is

$$c_x = \frac{\text{col} + 0.5}{g}, \qquad c_y = \frac{\text{row} + 0.5}{g}$$

**The order is important**: grid by grid (8, then 4, 2, 1); inside a grid, row by row (like
reading a page); inside a cell, the 4 boxes in the order of the table. Each default box is stored in **center** format
`[cx, cy, w, h]`.

**TODO 2:** fill in `make_default_boxes`.

In [ ]:
GRIDS = [8, 4, 2, 1]
SMALL = [0.1, 0.3, 0.5, 0.7]
LARGE = [0.2, 0.4, 0.6, 0.8]


def make_default_boxes(grids, small, large):
    boxes = []
    for g, s, l in zip(grids, small, large):
        for row in range(g):
            for col in range(g):
                # TODO: the center of cell (row, col)
                cx = ...
                cy = ...
                # TODO: append the 4 default boxes of this cell, each as [cx, cy, w, h], in the order of the table
                boxes.append([cx, cy, s, s])



    return np.array(boxes, dtype=np.float32)

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

GRIDS = [8, 4, 2, 1]
SMALL = [0.1, 0.3, 0.5, 0.7]
LARGE = [0.2, 0.4, 0.6, 0.8]


def make_default_boxes(grids, small, large):
    boxes = []
    for g, s, l in zip(grids, small, large):
        for row in range(g):
            for col in range(g):
                cx = (col + 0.5) / g
                cy = (row + 0.5) / g
                boxes.append([cx, cy, s, s])
                boxes.append([cx, cy, l, l])
                boxes.append([cx, cy, l * math.sqrt(2), l / math.sqrt(2)])
                boxes.append([cx, cy, l / math.sqrt(2), l * math.sqrt(2)])
    return np.array(boxes, dtype=np.float32)

In [ ]:
# run this cell to check your code
DEFAULT = make_default_boxes(GRIDS, SMALL, LARGE)
print("shape:", DEFAULT.shape)
print("the 4 boxes of the first cell:\n", DEFAULT[:4].round(4))
ok = DEFAULT.shape == (340, 4)
ok = ok and np.allclose(DEFAULT[:4], [[0.0625, 0.0625, 0.1, 0.1], [0.0625, 0.0625, 0.2, 0.2],
                                      [0.0625, 0.0625, 0.28284, 0.14142], [0.0625, 0.0625, 0.14142, 0.28284]], atol=1e-4)
ok = ok and np.allclose(DEFAULT[4 * 9], [0.1875, 0.1875, 0.1, 0.1])         # grid 8, row 1, col 1
ok = ok and np.allclose(DEFAULT[256 + 4 * 6 + 1], [0.625, 0.375, 0.4, 0.4])   # grid 4, row 1, col 2, large square
ok = ok and np.allclose(DEFAULT[-1], [0.5, 0.5, 0.56569, 1.13137], atol=1e-4)
print("correct:", ok)

Two helpers: switch between **center** `[x, y, w, h]` and **corner** `[x1, y1, x2, y2]` format (Lab 6, TODO 1), for many
boxes at once. They work for NumPy arrays and for torch tensors.

Some default boxes stick out of the image (the large square of the first cell goes from $-0.0375$ to $0.1625$). We **clip** their
corners to the image (0 to 1) for matching.

In [ ]:
def to_corners(b):
    # (N, 4) center format -> (N, 4) corner format
    x, y, w, h = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    out = [x - w / 2, y - h / 2, x + w / 2, y + h / 2]
    return torch.stack(out, dim=1) if torch.is_tensor(b) else np.stack(out, axis=1)


def to_center(b):
    # (N, 4) corner format -> (N, 4) center format
    x1, y1, x2, y2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    out = [(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]
    return torch.stack(out, dim=1) if torch.is_tensor(b) else np.stack(out, axis=1)


DEFAULT_CORNERS = np.clip(to_corners(DEFAULT), 0, 1)
GRID_OF_BOX = np.concatenate([[i] * g * g * 4 for i, g in enumerate(GRIDS)])   # which grid every default box belongs to
print("large square of the first cell, corners:", to_corners(DEFAULT)[1], " clipped:", DEFAULT_CORNERS[1])

Here are the 4 default boxes of one cell in every grid. **Try it:** change `row` and `col`.

In [ ]:
row, col = 1, 1
BOX_COLORS = ["red", "lime", "yellow", "cyan"]   # small square, large square, wide, tall

fig, axes = plt.subplots(1, 4, figsize=(16, 4.3))
start = 0
for ax, g in zip(axes, GRIDS):
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)
    ax.set_aspect("equal")
    for k in range(1, g):
        ax.axhline(k / g, color="gray", linewidth=0.5)
        ax.axvline(k / g, color="gray", linewidth=0.5)
    r, c = min(row, g - 1), min(col, g - 1)
    first = start + (r * g + c) * 4            # index of the first default box of this cell
    for b in range(4):
        x1, y1, x2, y2 = to_corners(DEFAULT[first + b : first + b + 1])[0]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=BOX_COLORS[b], linewidth=2))
    ax.add_patch(patches.Rectangle((c / g, r / g), 1 / g, 1 / g, color="blue", alpha=0.25))
    ax.set_title(f"grid {g}x{g}, cell ({r}, {c}): boxes {first} to {first + 3}")
    start += g * g * 4
plt.show()

## 4. Matching and offsets (A3 section 4)

For training, every one of the 340 default boxes needs a **target**: a class (cat, dog, person or **background**) and, if it
carries an object, the box **offsets**.

### IoU

To decide which default boxes carry an object we use IoU (you wrote it in Lab 6, TODO 4). This version compares **one** box
with **many** boxes at once, with NumPy.

In [ ]:
def iou(boxes, box):
    # boxes: (N, 4) corners, box: (4,) corners. Returns (N,) the IoU of every box with box
    left = np.maximum(boxes[:, 0], box[0])
    top = np.maximum(boxes[:, 1], box[1])
    right = np.minimum(boxes[:, 2], box[2])
    bottom = np.minimum(boxes[:, 3], box[3])
    intersection = np.maximum(right - left, 0) * np.maximum(bottom - top, 0)
    area1 = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    area2 = (box[2] - box[0]) * (box[3] - box[1])
    return intersection / (area1 + area2 - intersection)


boxes = np.array([[0.1, 0.1, 0.6, 0.6], [0.0, 0.0, 0.2, 0.2]])
box = np.array([0.3, 0.3, 0.8, 0.8])
ious = iou(boxes, box)
print(ious)   # Lab 6 example: 0.22 and 0

# fig, ax = plt.subplots(figsize=(4, 4))
# ax.add_patch(patches.Rectangle(box[:2], box[2] - box[0], box[3] - box[1], color="tab:orange", alpha=0.4, label="box"))
# for b, value, color in zip(boxes, ious, ["tab:blue", "tab:green"]):
#     ax.add_patch(patches.Rectangle(b[:2], b[2] - b[0], b[3] - b[1], fill=False, edgecolor=color, linewidth=2, label=f"IoU = {value:.2f}"))
# ax.set_xlim(0, 1)
# ax.set_ylim(1, 0)   # y goes down, like in an image
# ax.set_aspect("equal")
# ax.legend()
# plt.show()

### Offsets: the correction to a default box

The network does **not** predict the box directly. It predicts how to move and stretch a default box $p = (p_x, p_y, p_w, p_h)$
to get the true box $g = (g_x, g_y, g_w, g_h)$ (both in center format).

**encode**: true box → offsets $t$ (the target the network learns):

$$t_x = \frac{g_x - p_x}{p_w}, \qquad t_y = \frac{g_y - p_y}{p_h}, \qquad t_w = \log\frac{g_w}{p_w}, \qquad t_h = \log\frac{g_h}{p_h}$$

- The shift is divided by the size of the default box: "move by half your width" means the same for big and small boxes.
- $\log$: if the true box is the same size, $t_w = 0$; twice as wide, $t_w = 0.69$; half as wide, $t_w = -0.69$.
- If the default box is already perfect, all offsets are **0**. The network only has to learn small numbers.

**decode**: the network's predicted offsets $d$ → a box (the opposite of encode):

$$\hat g_x = p_w d_x + p_x, \qquad \hat g_y = p_h d_y + p_y, \qquad \hat g_w = p_w e^{d_w}, \qquad \hat g_h = p_h e^{d_h}$$

**TODO 3:** fill in `encode` (one box, NumPy) and `decode` (many boxes at once, torch).

In [ ]:
def encode(g, p):
    # g: true box, p: default box, both [x, y, w, h] (center format). Returns the 4 offsets
    gx, gy, gw, gh = g
    px, py, pw, ph = p
    # TODO: the 4 formulas above (np.log for the log)
    tx = ...
    ty = ...
    tw = ...
    th = ...
    return np.array([tx, ty, tw, th], dtype=np.float32)


def decode(d, p):
    # d: (N, 4) offsets, p: (N, 4) default boxes (torch tensors). Returns (N, 4) boxes [x, y, w, h]
    dx, dy, dw, dh = d[:, 0], d[:, 1], d[:, 2], d[:, 3]
    px, py, pw, ph = p[:, 0], p[:, 1], p[:, 2], p[:, 3]
    x = pw * dx + px
    # TODO: y, w, h (torch.exp for e^)
    y = ...
    w = ...
    h = ...
    return torch.stack([x, y, w, h], dim=1)

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def encode(g, p):
    # g: true box, p: default box, both [x, y, w, h] (center format). Returns the 4 offsets
    gx, gy, gw, gh = g
    px, py, pw, ph = p
    tx = (gx - px) / pw
    ty = (gy - py) / ph
    tw = np.log(gw / pw)
    th = np.log(gh / ph)
    return np.array([tx, ty, tw, th], dtype=np.float32)


def decode(d, p):
    # d: (N, 4) offsets, p: (N, 4) default boxes (torch tensors). Returns (N, 4) boxes [x, y, w, h]
    dx, dy, dw, dh = d[:, 0], d[:, 1], d[:, 2], d[:, 3]
    px, py, pw, ph = p[:, 0], p[:, 1], p[:, 2], p[:, 3]
    x = pw * dx + px
    y = ph * dy + py
    w = pw * torch.exp(dw)
    h = ph * torch.exp(dh)
    return torch.stack([x, y, w, h], dim=1)

In [ ]:
# run this cell to check your code
t = encode([0.5, 0.4, 0.4, 0.1], [0.4, 0.5, 0.2, 0.2])
print("offsets:", t.round(4))
ok = np.allclose(t, [0.5, -0.5, 0.6931, -0.6931], atol=1e-4)

# round trip: random true boxes -> offsets -> boxes again
rng = np.random.default_rng(0)
g = rng.uniform(0.1, 0.9, (340, 4)).astype(np.float32)
d = np.stack([encode(g[i], DEFAULT[i]) for i in range(340)])
back = decode(torch.tensor(d), torch.tensor(DEFAULT)).numpy()
ok = ok and np.allclose(back, g, atol=1e-5)
print("correct:", ok)

### Which default boxes carry an object?

The rule, for every true object:
1. every default box with **IoU > 0.5** with the object carries it;
2. if no default box has IoU > 0.5, use the **one with the largest IoU** anyway, so that every object has at least one box.

A default box that carries an object gets the object's **class** and the **offsets** from the default box to the object.
All other default boxes are **background** (class 3), and their offsets are not used.

*We store the class as a number (1 = dog), not a one-hot vector (`[0, 1, 0, 0]`), because `F.cross_entropy` wants
numbers. `one_hot.argmax()` turns one into the other.*

**TODO 4:** fill in `match`. Use the **clipped** default boxes `DEFAULT_CORNERS` for the IoU, and the unclipped `DEFAULT` for the offsets.

In [ ]:
NUM_CLASSES = 4        # cat, dog, person, background
BACKGROUND = 3
THRESHOLD = 0.5


def match(gt_boxes, gt_labels):
    # gt_boxes: (n, 4) corners (normalized), gt_labels: n class numbers
    # returns labels: (340,) the class of every default box, offsets: (340, 4)
    labels = np.full(len(DEFAULT), BACKGROUND, dtype=np.int64)   # everything starts as background
    offsets = np.zeros((len(DEFAULT), 4), dtype=np.float32)
    for box, c in zip(gt_boxes, gt_labels):
        ious = iou(DEFAULT_CORNERS, box)
        # TODO: the indices of the default boxes with IoU > THRESHOLD (hint: np.where(...)[0].tolist())
        chosen = ...
        # TODO: add the default box with the largest IoU (hint: int(np.argmax(...)))
        best = ...
        chosen.append(best)
        g = to_center(box[None])[0]          # the true box in center format
        for i in chosen:
            # TODO: default box i carries this object: set its label and its offsets (use encode)
            labels[i] = ...
            offsets[i] = ...
    return labels, offsets

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

NUM_CLASSES = 4        # cat, dog, person, background
BACKGROUND = 3
THRESHOLD = 0.5


def match(gt_boxes, gt_labels):
    # gt_boxes: (n, 4) corners (normalized), gt_labels: n class numbers
    # returns labels: (340,) the class of every default box, offsets: (340, 4)
    labels = np.full(len(DEFAULT), BACKGROUND, dtype=np.int64)   # everything starts as background
    offsets = np.zeros((len(DEFAULT), 4), dtype=np.float32)
    for box, c in zip(gt_boxes, gt_labels):
        ious = iou(DEFAULT_CORNERS, box)
        chosen = np.where(ious > THRESHOLD)[0].tolist()
        best = int(np.argmax(ious))
        chosen.append(best)
        g = to_center(box[None])[0]          # the true box in center format
        for i in chosen:
            labels[i] = c
            offsets[i] = encode(g, DEFAULT[i])
    return labels, offsets

In [ ]:
# run this cell to check your code
labels, offsets = match(np.array([[0.1, 0.3, 0.5, 0.9]]), [1])
print("default boxes with a dog:", np.where(labels == 1)[0].tolist())
ok = np.where(labels == 1)[0].tolist() == [295, 331] and (labels == BACKGROUND).sum() == 338
ok = ok and np.allclose(offsets[295], [-0.265165, -0.044194, 0.346574, 0.058891], atol=1e-4)
labels, offsets = match(np.array([[0.40, 0.40, 0.43, 0.43]]), [0])   # a tiny cat: no IoU > 0.5, so only the best box
print("default boxes with a tiny cat:", np.where(labels == 0)[0].tolist())
ok = ok and np.where(labels == 0)[0].tolist() == [108]
print("correct:", ok)

Which grid takes which object? Below: the true box (dashed white) and the default boxes that carry it, colored by grid.

In [ ]:
GRID_COLORS = ["magenta", "cyan", "yellow", "orange"]

examples = [2, 5, 1, 4, 25, 57, 0, 39, 3, 14]     # 2 images per grid (8, 4, 2, 1), then 2 that use two grids
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for ax, k in zip(axes.flat, examples):
    name = train_names[k]
    image, boxes, labels = load(name)
    box_labels, _ = match(boxes, labels)
    ax.imshow(image)
    for i in np.where(box_labels != BACKGROUND)[0]:
        x1, y1, x2, y2 = DEFAULT_CORNERS[i] * IMG
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=GRID_COLORS[GRID_OF_BOX[i]], linewidth=2))
    for x1, y1, x2, y2 in boxes * IMG:
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="white", linewidth=2, linestyle="--"))
    ax.set_title(f"{(box_labels != BACKGROUND).sum()} default boxes")
    ax.axis("off")
handles = [patches.Patch(color=GRID_COLORS[i], label=f"grid {g}x{g}") for i, g in enumerate(GRIDS)]
fig.legend(handles=handles, loc="lower center", ncol=4)
plt.show()

# over the whole training set: how many objects does each grid carry?
counts = np.zeros(len(GRIDS))
for name in train_names[:1000]:
    image, boxes, labels = load(name)
    box_labels, _ = match(boxes, labels)
    for i in np.where(box_labels != BACKGROUND)[0]:
        counts[GRID_OF_BOX[i]] += 1
plt.bar([f"{g}x{g}" for g in GRIDS], counts, color=GRID_COLORS)
plt.ylabel("matched default boxes (1000 images)")
plt.show()

Most objects in this dataset are big, so the coarse grids (4×4 and 2×2) do most of the work. A small object would be caught
by the 8×8 grid. *(You can change `SMALL` and `LARGE` to fit the data better.)*

### Data augmentation: move the boxes too!

Detection needs data augmentation too. The rule: **whatever you do to the
image, do it to the boxes too.** Here is the easiest one, a horizontal flip: $x$ becomes $1 - x$, so the left edge of the new box
is $1 - $ (old right edge).

**TODO 5:** fill in `hflip`.

*(For a random crop: cut the image, then shift the boxes by the crop corner, divide by the crop size and clip them to 0..1.
Be careful with boxes that end up (almost) outside the crop.)*

In [ ]:
def hflip(image, boxes):
    # image: (H, W, 3) array, boxes: (n, 4) corners (normalized)
    image = image[:, ::-1].copy()
    # TODO: the flipped boxes. new x1 = 1 - old x2, new x2 = 1 - old x1, y1 and y2 stay
    x1 = ...
    x2 = ...
    new_boxes = np.stack([x1, boxes[:, 1], x2, boxes[:, 3]], axis=1)
    return image, new_boxes

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def hflip(image, boxes):
    # image: (H, W, 3) array, boxes: (n, 4) corners (normalized)
    image = image[:, ::-1].copy()
    x1 = 1 - boxes[:, 2]
    x2 = 1 - boxes[:, 0]
    new_boxes = np.stack([x1, boxes[:, 1], x2, boxes[:, 3]], axis=1)
    return image, new_boxes

In [ ]:
# run this cell to check your code
image, boxes, labels = load(train_names[3])
flipped, new_boxes = hflip(image, boxes)
fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(image)
draw_boxes(axes[0], boxes, labels)
axes[1].imshow(flipped)
draw_boxes(axes[1], new_boxes, labels)
for ax in axes:
    ax.axis("off")
plt.show()
_, b = hflip(image, np.array([[0.1, 0.2, 0.3, 0.4]]))
print("correct:", np.allclose(b, [[0.7, 0.2, 0.9, 0.4]]))

### The Dataset

Everything together: read the image and boxes, augment, match.
The images are small, so we read them all once at the start. We normalize the image with the ImageNet mean and std, because
our network starts with a ResNet18 trained on ImageNet.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


class TinyCOCO(Dataset):
    def __init__(self, names, train):
        self.train = train
        self.data = [load(name) for name in names]     # (image, boxes, labels) for every image

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        image, boxes, labels = self.data[i]
        if self.train and random.random() < 0.5:       # augmentation only for training
            image, boxes = hflip(image, boxes)
        box_labels, offsets = match(boxes, labels)
        x = torch.tensor(image).permute(2, 0, 1).float() / 255   # (3, 128, 128), 0..1
        x = (x - MEAN) / STD
        return x, torch.tensor(offsets), torch.tensor(box_labels)


train_set = TinyCOCO(train_names, train=True)
val_set = TinyCOCO(val_names, train=False)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64)
x, offsets, box_labels = train_set[0]
print("image:", x.shape, " offsets:", offsets.shape, " labels:", box_labels.shape)

## 5. The network (A3 section 5)

1. **Backbone:** a pretrained ResNet18 (Lab 5 and 7), cut after `layer3`: $3 \times 128 \times 128 \to 256 \times 8 \times 8$.
2. **Three "down" blocks** (a 1×1 conv, then a 3×3 conv with stride 2):
   $8 \times 8 \to 4 \times 4 \to 2 \times 2 \to 1 \times 1$. Now we have **4 feature maps**, one per grid.
3. **Two heads on every map**, both 3×3 convs:
   - class head: 4 boxes × 4 classes = **16 channels**,
   - box head: 4 boxes × 4 offsets = **16 channels**.

Tips: the class head and the box head are **separate** layers, every head is its **own** `nn.Conv2d` (even if the shapes
are the same), and there is **no softmax** in `forward` (`F.cross_entropy` does the softmax itself).

### From a head output to a list of boxes

A head gives a tensor of shape $(N, 16, g, g)$. We want $(N, g \cdot g \cdot 4, 4)$: one row per default box, **in the same order
as `DEFAULT`** (row by row, then the 4 boxes of the cell).

1. `permute(0, 2, 3, 1)`: move the channels to the end: $(N, g, g, 16)$. Now the cells are in reading order.
2. `reshape(N, g * g * 4, 4)`: cut the 16 numbers of every cell into 4 boxes of 4 numbers.

*Doing `reshape` **without** `permute`
first gives the right shape but the wrong order: the network still trains, but box $i$ no longer belongs to default box $i$,
and the results are bad.*

**TODO 6:** fill in `flatten`.

In [ ]:
def flatten(head_output):
    # (N, 16, g, g) -> (N, g * g * 4, 4)
    N = head_output.shape[0]
    # TODO: permute, then reshape (-1 lets torch compute g * g * 4)
    x = ...
    x = ...
    return x

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def flatten(head_output):
    # (N, 16, g, g) -> (N, g * g * 4, 4)
    N = head_output.shape[0]
    x = head_output.permute(0, 2, 3, 1)
    x = x.reshape(N, -1, 4)
    return x

In [ ]:
# run this cell to check your code
h = torch.randn(2, 16, 4, 4)
out = flatten(h)
print("shape:", out.shape)
ok = out.shape == (2, 64, 4)
for n in range(2):                     # compare with the rule, one number at a time
    for row in range(4):
        for col in range(4):
            for b in range(4):
                for k in range(4):
                    ok = ok and out[n, (row * 4 + col) * 4 + b, k] == h[n, b * 4 + k, row, col]
print("correct:", bool(ok))

**What `flatten` did**, for a 2×2 grid, for each head. Colors = cells. Each cell's 16 channels (left) become 4 rows of the
list (right): **one row = one default box**, and its 4 columns are the 4 class scores (class head) or the 4 offsets (box head).

In [ ]:
# a picture of flatten for a 2 x 2 grid (1 image), once for each head. Every number gets an id: cell number * 16 + channel
g = 2
h = torch.zeros(1, 16, g, g)
for row in range(g):
    for col in range(g):
        for ch in range(16):
            h[0, ch, row, col] = (row * g + col) * 16 + ch
out = flatten(h)[0]                       # (g * g * 4, 4) = (16, 4)
CELL_COLORS = ["#f4a6a6", "#a6c8f4", "#b5e3a1", "#f6d68a"]


def show_flatten(names, head):
    # names: what the 4 numbers k of a box mean in this head
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    ax = axes[0]                          # left: the head output, 16 channels in every cell
    for row in range(g):
        for col in range(g):
            cell = row * g + col
            for b in range(4):
                for k in range(4):
                    x = col * 5 + k
                    y = row * 5 + b
                    ax.add_patch(patches.Rectangle((x, y), 1, 1, facecolor=CELL_COLORS[cell], edgecolor="black"))
                    ax.text(x + 0.5, y + 0.5, f"ch {b * 4 + k}\n{names[k]}", ha="center", va="center", fontsize=7)
            ax.text(col * 5 + 2, row * 5 - 0.3, f"cell ({row}, {col})", ha="center", fontsize=10, fontweight="bold")
    ax.set_xlim(-0.5, 9.5)
    ax.set_ylim(9.5, -1)
    ax.set_title(f"{head} output (1, 16, 2, 2)\nin every cell: one row per box, 4 numbers per box")
    ax.axis("off")

    ax = axes[1]                          # right: the list after flatten
    for k in range(4):
        ax.text(k + 0.5, -0.3, names[k], ha="center", fontsize=9, fontweight="bold")
    for i in range(out.shape[0]):
        for k in range(4):
            cell = int(out[i, k]) // 16
            ch = int(out[i, k]) % 16
            ax.add_patch(patches.Rectangle((k, i), 1, 1, facecolor=CELL_COLORS[cell], edgecolor="black"))
            ax.text(k + 0.5, i + 0.5, f"ch {ch}", ha="center", va="center", fontsize=8)
        cell = int(out[i, 0]) // 16
        ax.text(-0.2, i + 0.5, f"row {i:2d} = cell ({cell // g}, {cell % g}), box {i % 4}", ha="right", va="center", fontsize=8)
    ax.set_xlim(-4.5, 4.2)
    ax.set_ylim(16, -1)
    ax.set_title("after flatten (1, 16, 4): one row per default box")
    ax.axis("off")
    plt.show()


show_flatten(["cat", "dog", "person", "backgr."], "class head")
show_flatten(["dx", "dy", "dw", "dh"], "box head")

**TODO 7:** fill in `forward`. The layers are ready in `__init__`. The grids must be concatenated in the order
8, 4, 2, 1, like `DEFAULT`.

In [ ]:
class Head(nn.Conv2d):
    # a normal conv layer. The extra line only fixes a bug of old PyTorch versions (< 2.5) on Mac GPUs
    def forward(self, x):
        y = super().forward(x)
        if y.requires_grad:
            y.register_hook(lambda grad: grad.contiguous())
        return y


def down_block(channels):
    # halves the size: g x g -> g/2 x g/2 (1x1 -> stays 1x1)
    return nn.Sequential(
        nn.Conv2d(channels, channels, kernel_size=1), nn.BatchNorm2d(channels), nn.ReLU(),
        nn.Conv2d(channels, channels, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(channels), nn.ReLU(),
    )


class TinySSD(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = torchvision.models.resnet18(weights="DEFAULT")
        self.backbone = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
                                      resnet.layer1, resnet.layer2, resnet.layer3)     # -> 256 x 8 x 8
        self.down1 = down_block(256)    # -> 256 x 4 x 4
        self.down2 = down_block(256)    # -> 256 x 2 x 2
        self.down3 = down_block(256)    # -> 256 x 1 x 1
        self.class_heads = nn.ModuleList([Head(256, 4 * NUM_CLASSES, kernel_size=3, padding=1) for _ in GRIDS])
        self.box_heads = nn.ModuleList([Head(256, 4 * 4, kernel_size=3, padding=1) for _ in GRIDS])

    def forward(self, x):
        f1 = self.backbone(x)
        # TODO: the other 3 feature maps
        f2 = ...
        f3 = ...
        f4 = ...
        maps = [f1, f2, f3, f4]
        confs = []
        locs = []
        for i in range(len(maps)):
            # TODO: class head i and box head i on maps[i], then flatten, then append to confs and locs
            pass


        # TODO: concatenate the 4 grids along dim 1 -> (N, 340, 4)
        conf = ...
        loc = ...
        return conf, loc        # raw class scores (no softmax!) and offsets

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

class Head(nn.Conv2d):
    # a normal conv layer. The extra line only fixes a bug of old PyTorch versions (< 2.5) on Mac GPUs
    def forward(self, x):
        y = super().forward(x)
        if y.requires_grad:
            y.register_hook(lambda grad: grad.contiguous())
        return y


def down_block(channels):
    # halves the size: g x g -> g/2 x g/2 (1x1 -> stays 1x1)
    return nn.Sequential(
        nn.Conv2d(channels, channels, kernel_size=1), nn.BatchNorm2d(channels), nn.ReLU(),
        nn.Conv2d(channels, channels, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(channels), nn.ReLU(),
    )


class TinySSD(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = torchvision.models.resnet18(weights="DEFAULT")
        self.backbone = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
                                      resnet.layer1, resnet.layer2, resnet.layer3)     # -> 256 x 8 x 8
        self.down1 = down_block(256)    # -> 256 x 4 x 4
        self.down2 = down_block(256)    # -> 256 x 2 x 2
        self.down3 = down_block(256)    # -> 256 x 1 x 1
        self.class_heads = nn.ModuleList([Head(256, 4 * NUM_CLASSES, kernel_size=3, padding=1) for _ in GRIDS])
        self.box_heads = nn.ModuleList([Head(256, 4 * 4, kernel_size=3, padding=1) for _ in GRIDS])

    def forward(self, x):
        f1 = self.backbone(x)
        f2 = self.down1(f1)
        f3 = self.down2(f2)
        f4 = self.down3(f3)
        maps = [f1, f2, f3, f4]
        confs = []
        locs = []
        for i in range(len(maps)):
            confs.append(flatten(self.class_heads[i](maps[i])))
            locs.append(flatten(self.box_heads[i](maps[i])))
        conf = torch.cat(confs, dim=1)
        loc = torch.cat(locs, dim=1)
        return conf, loc        # raw class scores (no softmax!) and offsets

In [ ]:
# run this cell to check your code
torch.manual_seed(0)
m = TinySSD()
conf, loc = m(torch.zeros(2, 3, IMG, IMG))
n_params = sum(p.numel() for p in m.parameters())
print("conf:", conf.shape, " loc:", loc.shape, " parameters:", n_params)
ok = conf.shape == (2, 340, 4) and loc.shape == (2, 340, 4) and n_params == 5048512
# are the last 4 boxes (the 1x1 grid) really computed by the last box head?
m.eval()
x = torch.randn(1, 3, IMG, IMG)
with torch.no_grad():
    f4 = m.down3(m.down2(m.down1(m.backbone(x))))
    ok = ok and torch.allclose(m(x)[1][0, 336:], flatten(m.box_heads[3](f4))[0], atol=1e-5)
print("correct:", bool(ok))

## 6. Loss and training (A3 section 6)

The loss. Boxes **with** an object ($x^{obj}_i = 1$) and boxes **without** ($x^{noobj}_i = 1$, background) are averaged separately:

$$
L_{cls} = \frac{1}{\sum_i x^{obj}_i} \sum_i x^{obj}_i \,\text{CE}(\text{conf}^{pred}_i, \text{conf}^{gt}_i)
\;+\; 3 \cdot \frac{1}{\sum_i x^{noobj}_i} \sum_i x^{noobj}_i \,\text{CE}(\text{conf}^{pred}_i, \text{conf}^{gt}_i)
$$
$$
L_{box} = \frac{1}{\sum_i x^{obj}_i} \sum_i x^{obj}_i \,\text{SmoothL1}(\text{box}^{pred}_i, \text{box}^{gt}_i)
\qquad\qquad L = L_{cls} + L_{box}
$$

- "Average over the boxes with an object" is just `F.cross_entropy(conf[obj], labels[obj])`: the built-in functions average for you.
- Background boxes are about 336 of 340, but they get their **own** average, so they cannot drown the few object boxes.
  The factor 3 says "saying *nothing here* correctly matters too".
- The box loss is only for boxes **with** an object: for background, there is no box to predict.
- Smooth L1 is like $|x|$, but round near 0 (so the gradient does not jump).

Once you have the mask, the loss is only a few lines.

**TODO 8:** fill in `ssd_loss`. Use the boolean mask `obj`, not `torch.nonzero`.

In [ ]:
def ssd_loss(conf, loc, labels, offsets):
    # conf: (N, 340, 4) raw class scores, loc: (N, 340, 4) predicted offsets
    # labels: (N, 340) class numbers (3 = background), offsets: (N, 340, 4) true offsets
    conf = conf.reshape(-1, NUM_CLASSES)    # put all boxes of all images in one long list
    loc = loc.reshape(-1, 4)
    labels = labels.reshape(-1)
    offsets = offsets.reshape(-1, 4)
    obj = labels != BACKGROUND              # True for boxes that carry an object
    # TODO: cross-entropy on the boxes with an object + 3 * cross-entropy on the background boxes (~obj)
    class_loss = ...
    # TODO: smooth L1 (F.smooth_l1_loss) between predicted and true offsets, only boxes with an object
    box_loss = ...
    return class_loss + box_loss

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def ssd_loss(conf, loc, labels, offsets):
    # conf: (N, 340, 4) raw class scores, loc: (N, 340, 4) predicted offsets
    # labels: (N, 340) class numbers (3 = background), offsets: (N, 340, 4) true offsets
    conf = conf.reshape(-1, NUM_CLASSES)    # put all boxes of all images in one long list
    loc = loc.reshape(-1, 4)
    labels = labels.reshape(-1)
    offsets = offsets.reshape(-1, 4)
    obj = labels != BACKGROUND              # True for boxes that carry an object
    class_loss = F.cross_entropy(conf[obj], labels[obj]) + 3 * F.cross_entropy(conf[~obj], labels[~obj])
    box_loss = F.smooth_l1_loss(loc[obj], offsets[obj])
    return class_loss + box_loss

In [ ]:
# run this cell to check your code
g = torch.Generator().manual_seed(0)
conf = torch.randn(2, 340, 4, generator=g)
loc = torch.randn(2, 340, 4, generator=g)
labels = torch.full((2, 340), BACKGROUND)
labels[0, [10, 11, 300]] = torch.tensor([0, 0, 2])
labels[1, 5] = 1
offsets = torch.randn(2, 340, 4, generator=g)
value = ssd_loss(conf, loc, labels, offsets).item()
print("loss:", round(value, 4))
print("correct:", abs(value - 7.6785) < 1e-3)

### Train

The usual loop (Lab 7): forward → loss → backward → update, and the validation loss after every epoch.

**How long?** About 1 minute for 10 epochs on a GPU (Colab, or a Mac). On a laptop **without** a GPU one epoch takes
1 to 2 minutes: set `EPOCHS = 2`, watch the loss go down, and then load our model trained for 10 epochs in the next cell.

In [ ]:
EPOCHS = 10


def run_epoch(model, loader, loss_fn, optimizer=None):
    # one pass over the data. With an optimizer: train. Without: just compute the loss
    model.train(optimizer is not None)
    total = 0
    for x, offsets, labels in loader:
        x, offsets, labels = x.to(device), offsets.to(device), labels.to(device)
        with torch.set_grad_enabled(optimizer is not None):
            conf, loc = model(x)
            loss = loss_fn(conf, loc, labels, offsets)
        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total += loss.item() * len(x)
    return total / len(loader.dataset)


def train_model(loss_fn, epochs):
    torch.manual_seed(0)
    random.seed(0)
    model = TinySSD().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)   # lr goes smoothly down to 0
    history = []
    for epoch in range(epochs):
        train_loss = run_epoch(model, train_loader, loss_fn, optimizer)
        val_loss = run_epoch(model, val_loader, loss_fn)
        scheduler.step()
        history.append((train_loss, val_loss))
        print(f"epoch {epoch + 1:2d} | train loss {train_loss:.3f} | val loss {val_loss:.3f}")
    return model, history


model, history = train_model(ssd_loss, EPOCHS)
torch.save(model.state_dict(), "my_tiny_ssd.pth")

plt.plot(range(1, EPOCHS + 1), [h[0] for h in history], label="train")
plt.plot(range(1, EPOCHS + 1), [h[1] for h in history], label="validation")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

In [ ]:
USE_OUR_MODEL = False     # set to True if you trained only a few epochs (no GPU)

if USE_OUR_MODEL:
    if not os.path.exists("tiny_ssd.pth"):
        urllib.request.urlretrieve(RAW + "lab9/tiny_ssd.pth", "tiny_ssd.pth")   # 10 MB, trained with the code above
    state = torch.load("tiny_ssd.pth", map_location="cpu")
    model = TinySSD()
    model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in state.items()})
    model = model.to(device)
model.eval()
print("ready")

The validation loss goes down much less than the training loss: we have only 2,466 training images, so the model overfits. Part of the validation loss can never go away: the **unlabeled** objects. The model is "punished" for finding them (you will see examples in Part 7).

## 7. From outputs to boxes (A3 section 7)

The network gives 340 class scores and 340 offsets. To get the final detections:

1. **softmax** over the 4 classes (only here, outside the model),
2. for every default box, its best **object** class (cat, dog or person) and that probability,
3. keep the boxes with probability > `threshold`,
4. **decode** the offsets with the default boxes, convert to corners,
5. **non-maximum suppression**: many default boxes overlap the same object, so they all fire. Keep the best one and
   remove boxes of the same class that overlap it by more than `overlap`.

You wrote NMS yourself in Lab 6 (TODO 7). Here we use
torchvision's `batched_nms` (NMS for each class separately).

**TODO 9:** fill in `detect`.

In [ ]:
DEFAULT_T = torch.tensor(DEFAULT)


def detect(conf, loc, threshold=0.5, overlap=0.45):
    # conf, loc: (340, 4) for ONE image, on the CPU
    probs = F.softmax(conf, dim=1)
    # TODO: the best object class of every box and its probability: max over the first 3 columns (hint: .max(dim=1))
    scores, classes = ...
    keep = scores > threshold
    scores, classes = scores[keep], classes[keep]
    # TODO: the predicted boxes of the kept default boxes: decode loc[keep] with DEFAULT_T[keep], then convert to corners
    boxes = ...
    boxes = boxes.clamp(0, 1)                    # stay inside the image
    k = torchvision.ops.batched_nms(boxes, scores, classes, overlap)
    from_box = keep.nonzero()[:, 0][k]           # which default box every detection came from
    return boxes[k], scores[k], classes[k], from_box

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

DEFAULT_T = torch.tensor(DEFAULT)


def detect(conf, loc, threshold=0.5, overlap=0.45):
    # conf, loc: (340, 4) for ONE image, on the CPU
    probs = F.softmax(conf, dim=1)
    scores, classes = probs[:, :3].max(dim=1)
    keep = scores > threshold
    scores, classes = scores[keep], classes[keep]
    boxes = to_corners(decode(loc[keep], DEFAULT_T[keep]))
    boxes = boxes.clamp(0, 1)                    # stay inside the image
    k = torchvision.ops.batched_nms(boxes, scores, classes, overlap)
    from_box = keep.nonzero()[:, 0][k]           # which default box every detection came from
    return boxes[k], scores[k], classes[k], from_box

In [ ]:
# run this cell to check your code
# a "perfect network": its outputs are the true targets of an image with a dog and a person
true_boxes = np.array([[0.1, 0.2, 0.5, 0.9], [0.55, 0.1, 0.95, 0.7]], dtype=np.float32)
box_labels, offsets = match(true_boxes, [1, 2])
fake_conf = F.one_hot(torch.tensor(box_labels), 4).float() * 10   # very sure of the right class
boxes, scores, classes, from_box = detect(fake_conf, torch.tensor(offsets))
print("boxes:", boxes.numpy().round(3).tolist(), " classes:", classes.tolist())
print("correct:", len(boxes) == 2 and np.allclose(sorted(boxes.numpy().tolist()), true_boxes, atol=1e-5) and sorted(classes.tolist()) == [1, 2])

### What does the network see?

Four views of the same validation image:
- top left: the **true** boxes; top right: the default boxes that **carry** them (what we trained on),
- bottom left: the **predicted** boxes; bottom right: the default boxes they **came from**.

A third row shows **all** boxes with probability > 0.5, **before** NMS.

In [ ]:
def predict(model, x):
    # x: (3, IMG, IMG) normalized image. Returns conf, loc for this image, on the CPU
    model.eval()
    with torch.no_grad():
        conf, loc = model(x[None].to(device))
    return conf[0].cpu(), loc[0].cpu()


fig, axes = plt.subplots(3, 4, figsize=(16, 12.5))
for j, i in enumerate([1, 15]):
    image, true_boxes, true_labels = val_set.data[i]
    x, _, box_labels = val_set[i]
    conf, loc = predict(model, x)
    boxes, scores, classes, from_box = detect(conf, loc)
    carry = np.where(box_labels.numpy() != BACKGROUND)[0]
    ax = axes[:, 2 * j : 2 * j + 2]
    for a in ax.flat:
        a.imshow(image)
        a.axis("off")
    draw_boxes(ax[0, 0], true_boxes, true_labels)
    draw_boxes(ax[0, 1], DEFAULT_CORNERS[carry], box_labels[carry], linestyle="--")
    draw_boxes(ax[1, 0], boxes, classes, scores)
    draw_boxes(ax[1, 1], DEFAULT_CORNERS[from_box.numpy()], classes, linestyle="--")
    many = detect(conf, loc, overlap=1.0)       # overlap 1.0 = no box is ever removed = no NMS
    draw_boxes(ax[2, 0], many[0], many[2])
    ax[0, 0].set_title("true boxes")
    ax[0, 1].set_title("default boxes that carry them")
    ax[1, 0].set_title("predicted boxes (after NMS)")
    ax[1, 1].set_title("default boxes they came from")
    ax[2, 0].set_title(f"before NMS: {len(many[0])} boxes")
    ax[2, 1].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# find validation images for 3 kinds of mistakes
duplicate, no_label, wrong_class = [], [], []
for i in range(len(val_set)):
    image, true_boxes, true_labels = val_set.data[i]
    x, _, _ = val_set[i]
    boxes, scores, classes, _ = detect(*predict(model, x))
    for t in range(len(true_boxes)):
        overlaps = iou(boxes.numpy(), true_boxes[t]) if len(boxes) else np.zeros(0)
        same = (overlaps > 0.3) & (classes.numpy() == true_labels[t])
        if same.sum() >= 2:
            duplicate.append(i)                    # two boxes of the right class on one object
        right = (overlaps >= 0.5) & (classes.numpy() == true_labels[t])
        wrong = (overlaps >= 0.5) & (classes.numpy() != true_labels[t])
        if wrong.any() and not right.any():
            wrong_class.append(i)                  # found the object, but called it the wrong class
    for j in range(len(boxes)):
        if scores[j] > 0.6 and iou(true_boxes, boxes[j].numpy()).max() < 0.2:
            no_label.append(i)                     # a sure box that overlaps no label at all
print(f"of {len(val_set)} validation images: {len(set(duplicate))} with two boxes for one object, "
      f"{len(set(no_label))} with a box but no label, {len(set(wrong_class))} with the wrong class")

rows = [("two boxes,\none object", duplicate), ("box with\nno label", no_label), ("wrong class", wrong_class)]
fig, axes = plt.subplots(3, 6, figsize=(18, 9.5))
for r, (name, images) in enumerate(rows):
    picked = sorted(set(images))[:6]
    for c in range(6):
        ax = axes[r, c]
        ax.axis("off")
        if c >= len(picked):
            continue
        image, true_boxes, true_labels = val_set.data[picked[c]]
        x, _, _ = val_set[picked[c]]
        boxes, scores, classes, _ = detect(*predict(model, x))
        if r == 1:                                 # this row: only draw the boxes without a label
            extra = [j for j in range(len(boxes)) if scores[j] > 0.6 and iou(true_boxes, boxes[j].numpy()).max() < 0.2]
            boxes, scores, classes = boxes[extra], scores[extra], classes[extra]
        ax.imshow(image)
        for x1, y1, x2, y2 in true_boxes * IMG:
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="white", linewidth=2, linestyle="--"))
        draw_boxes(ax, boxes, classes, scores)
    axes[r, 0].text(-10, IMG / 2, name, ha="right", va="center", fontsize=12, fontweight="bold")
fig.suptitle("validation images. dashed white = true box, colored = prediction (red cat, blue dog, green person)")
plt.tight_layout()
plt.show()

**The mistakes, one row each:**
- **Two boxes, one object:** they come from different default boxes and overlap less than `overlap`, so NMS keeps both.
  Try `overlap=0.3` in `detect`.
- **Box with no label** (only those boxes are drawn): many are real cats, dogs or people that nobody labeled (Part 2).
  The model is right, but the loss counts them as mistakes.
- **Wrong class:** mostly cat ↔ dog. Our images are only 128×128.

## 8. Try it yourself

The **test images** have no labels, so we can only look. Then our tiny SSD and the real SSDlite on the photo from Part 1.

In [ ]:
def detect_photo(model, photo, threshold=0.5):
    # photo: a PIL image of any size. Returns corner boxes (normalized), scores, classes
    image = np.array(photo.convert("RGB").resize((IMG, IMG)))
    x = (torch.tensor(image).permute(2, 0, 1).float() / 255 - MEAN) / STD
    boxes, scores, classes, _ = detect(*predict(model, x), threshold=threshold)
    return boxes, scores, classes


test_names = sorted(os.listdir(f"{DATA}/test/images"))
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for ax, name in zip(axes.flat, test_names):
    photo = Image.open(f"{DATA}/test/images/{name}").convert("RGB")
    boxes, scores, classes = detect_photo(model, photo)
    ax.imshow(photo.resize((IMG, IMG)))
    draw_boxes(ax, boxes, classes, scores)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
my_photo = "sopranos_dinner.jpg"        # try your own photo with cats, dogs or people!

photo = Image.open(my_photo).convert("RGB")
W, H = photo.size
boxes, scores, classes = detect_photo(model, photo, threshold=0.4)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
axes[0].imshow(photo)
for (x1, y1, x2, y2), s, c in zip(boxes.tolist(), scores.tolist(), classes.tolist()):
    axes[0].add_patch(patches.Rectangle((x1 * W, y1 * H), (x2 - x1) * W, (y2 - y1) * H, fill=False, edgecolor=COLORS[c], linewidth=2))
    axes[0].text(x1 * W, y1 * H - 4, f"{CLASSES[c]} {s:.2f}", color=COLORS[c], fontweight="bold")
axes[0].set_title("our tiny SSD (128 x 128, 340 default boxes)")
with torch.no_grad():
    result = real_ssd([torchvision.transforms.functional.to_tensor(photo)])[0]
axes[1].imshow(photo)
for box, score, label in zip(result["boxes"], result["scores"], result["labels"]):
    if score > 0.4 and COCO_NAMES[label] in CLASSES:
        x1, y1, x2, y2 = box.tolist()
        c = CLASSES.index(COCO_NAMES[label])
        axes[1].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=COLORS[c], linewidth=2))
        axes[1].text(x1, y1 - 4, f"{COCO_NAMES[label]} {score:.2f}", color=COLORS[c], fontweight="bold")
axes[1].set_title("real SSDlite (320 x 320, 3234 default boxes, 118k training images)")
for ax in axes:
    ax.axis("off")
plt.show()